In [1]:
# Standard library imports
import os
import sys
import math
import random
from collections import Counter, defaultdict
from typing import List
from random import sample as random_sample

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from tqdm import tqdm

# PyTorch imports
import torch
import torch.nn as nn
from torch import Tensor, no_grad, argmax
from torch import max as torch_max
from torch.nn import CrossEntropyLoss
from torch.optim import SGD, AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, CosineAnnealingWarmRestarts
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler
from torchsummary import summary
import torchvision.transforms as transforms
from torchvision.transforms import functional as F

# Ignite imports
from ignite.engine import Events, create_supervised_trainer, create_supervised_evaluator
from ignite.handlers import ReduceLROnPlateauScheduler
from ignite.metrics import Accuracy, Fbeta, Loss, Precision, Recall

In [2]:
import sys
import os

import torch
from torch.optim import SGD, AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, CosineAnnealingWarmRestarts
from torch.nn import CrossEntropyLoss
from torchsummary import summary

import src.dataloader as dataloader
from src.dataloader import PeopleDataset

from src.utils.engine import setup_trainer, setup_evaluators, train_epoch_and_get_metrics_dict, calculate_epoch_metrics
from src.utils.logging import setup_metrics_history, add_metrics_to_history, print_epoch_summary, save_best_models
from src.utils import plotting

def train_model(config, model_class, class_exclusion_threshold=None, classes_to_exclude=None):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}\n")

    """Preparing the data"""
    train_transforms = dataloader.get_transforms(augmentation_type=config.TRAIN_AUGMENTATION_TYPE)
    valid_transforms = dataloader.get_transforms(augmentation_type=config.VALID_AUGMENTATION_TYPE)

    print("Loading the dataset...")
    full_dataset = PeopleDataset(config.PATH_TO_DATA)
    full_dataset.print_class_distribution()

    if class_exclusion_threshold:
        print("Removing rare classes")
        # Option 1: Filter by minimum threshold of class in dataset
        full_dataset.filter_by_min_threshold(min_threshold=class_exclusion_threshold)

    if classes_to_exclude:
        print("Excluding selected classes")
        # Option 2: Filter by explicitly excluding class names
        full_dataset.filter_by_classes(classes_to_exclude=classes_to_exclude)

    if classes_to_exclude or class_exclusion_threshold:
        # Rebuild class_to_index AFTER filtering
        full_dataset.class_names = sorted(
            list(set(full_dataset.labels)))  # Get unique remaining labels (which are strings) and sort them
        full_dataset.class_to_index = {cls_name: i for i, cls_name in enumerate(full_dataset.class_names)}
        print(f"Number of classes after filtering: {len(full_dataset.class_names)}")  # Verify the number of classes

    train_set, valid_set = dataloader.split_dataset(full_dataset, valid_ratio=0.2)
    train_set.dataset.transform = train_transforms
    valid_set.dataset.transform = valid_transforms

    # Showing first 12 images after transforming them
    # plotting.show_first_images(full_dataset)

    print(f"Setting up data loaders with batch_size={config.BATCH_SIZE}...")
    train_loader, valid_loader = dataloader.setup_data_loaders(
        batch_size=config.BATCH_SIZE,
        train_set=train_set,
        valid_set=valid_set
    )

    """Training setup"""
    num_classes = len(full_dataset.class_names)  # Use the updated class_names
    print(f"Number of classes: {num_classes}")

    model = model_class(num_classes=num_classes)
    model.to(device)
    summary(model, (3, 288, 512))
    print("\n")

    train_indices = train_set.indices
    train_targets = [full_dataset[idx][1] for idx in train_indices]
    class_counts = [train_targets.count(i) for i in range(num_classes)]
    class_counts = [c if c != 0 else 1 for c in class_counts]
    class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
    class_weights = class_weights / class_weights.sum()
    class_weights = class_weights.to(device)

    optimizer = AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)

    if config.SCHEDULER == 'CosineAnnealingWarmRestarts':
        scheduler = CosineAnnealingWarmRestarts(optimizer=optimizer, T_0=25, eta_min=1e-6)
    elif config.SCHEDULER == 'CosineAnnealingLR':
        scheduler = CosineAnnealingLR(optimizer=optimizer, T_max=config.NUM_EPOCHS, eta_min=0)

    criterion = CrossEntropyLoss(weight=class_weights.to(device))

    print("Initializing trainer and evaluators...")
    trainer = setup_trainer(model, optimizer, criterion, device)
    train_evaluator, valid_evaluator = setup_evaluators(model, criterion, device)

    train_metrics_history, valid_metrics_history = setup_metrics_history()

    best_valid_loss = float('inf')
    best_valid_f1 = 0.0
    model_name = model.__class__.__name__

    print(f"\nStarting training for {config.NUM_EPOCHS} epochs...")

    for epoch in range(config.NUM_EPOCHS):
        print(f"\nEpoch {epoch + 1}/{config.NUM_EPOCHS}")

        train_metrics_dict = train_epoch_and_get_metrics_dict(model, train_loader, criterion, optimizer, device, epoch,
                                                              config.NUM_EPOCHS)
        scheduler.step()
        add_metrics_to_history(train_metrics_history, train_metrics_dict)

        valid_metrics_dict = {}
        if valid_loader:
            valid_metrics_dict = calculate_epoch_metrics(model, valid_loader, criterion, device)
            add_metrics_to_history(valid_metrics_history, valid_metrics_dict)
            best_valid_loss, best_valid_f1 = save_best_models(
                current_metrics=valid_metrics_dict,
                model=model,
                model_name=model_name,
                best_loss=best_valid_loss,
                best_f1=best_valid_f1,
                save_dir=config.SAVE_DIR
            )

        print_epoch_summary(epoch, train_metrics_dict, valid_metrics_dict)

    """Results visualization"""
    print("\nTraining completed!")
    print(f"Results location: {config.RESULT_DIR}")
    print("Saving results...")
    metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1', 'loss']
    plotting.plot_metrics(train_metrics_history, valid_metrics_history, metrics_to_plot, save_path=config.RESULT_DIR)

    # To plot loss and one metric
    # plot_metric_and_loss(train_metrics_history, valid_metrics_history, "accuracy")

    class_names = full_dataset.class_names  # Use the updated class_names
    # plotting.visualize_predictions(model, valid_loader, device, class_names)

    print(f"\nPlotting metrics per class...")
    plotting.plot_metrics_per_class(model, valid_loader, device, class_names, save_path=config.RESULT_DIR)

    # evaluate_model(model, test_loader, criterion, device)

In [ ]:
# simulating config file as class

class Config:
    def __init__(self):
        self.PATH_TO_DATA = "PATH_TO_YOUR_DATA"
        self.RESULT_DIR = "RESULTS_SAVING_DIRECTORY"
        self.SAVE_DIR = "WEIGHTS_SAING_DIRECTORY"

        self.BATCH_SIZE = 32
        self.LEARNING_RATE = 0.001
        self.MOMENTUM = 0.9
        self.NUM_EPOCHS = 20

        self.SCHEDULER = 'CosineAnnealingLR'

        self.WEIGHT_DECAY = 3e-3

        self.TRAIN_AUGMENTATION_TYPE = "basic"
        self.VALID_AUGMENTATION_TYPE = None

config = Config()

In [4]:
from src.models.__all_models import *

model_to_train = PoseCNNv2_Lite

# Call the training function from trainer.py
train_model(config, model_to_train)

Using device: cpu

Augmentation type: basic
Augmentation type: None
Loading the dataset...
Class Distribution:
  Class 'sports': 26 samples (0.2261)
  Class 'inactivity quiet/light': 2 samples (0.0174)
  Class 'miscellaneous': 7 samples (0.0609)
  Class 'occupation': 16 samples (0.1391)
  Class 'water activities': 8 samples (0.0696)
  Class 'home activities': 7 samples (0.0609)
  Class 'lawn and garden': 10 samples (0.0870)
  Class 'religious activities': 0 samples (0.0000)
  Class 'winter activities': 7 samples (0.0609)
  Class 'conditioning exercise': 5 samples (0.0435)
  Class 'bicycling': 3 samples (0.0261)
  Class 'fishing and hunting': 2 samples (0.0174)
  Class 'dancing': 4 samples (0.0348)
  Class 'walking': 3 samples (0.0261)
  Class 'running': 2 samples (0.0174)
  Class 'self care': 0 samples (0.0000)
  Class 'home repair': 10 samples (0.0870)
  Class 'volunteer activities': 0 samples (0.0000)
  Class 'music playing': 3 samples (0.0261)
  Class 'transportation': 0 samples (0.

Validation: 100%|██████████| 1/1 [00:21<00:00, 21.10s/batch, val_loss=2.98]


Saved new best model(s) - Loss: 2.9784

Epoch 1 Summary:
Train - Loss: 3.0590, Acc: 5.43%, Precision: 0.0998, Recall: 0.0543, F1: 0.0390
Valid - Loss: 2.9784, Acc: 0.00%, Precision: 0.0000, Recall: 0.0000, F1: 0.0000

Epoch 2/20


Validation: 100%|██████████| 1/1 [00:21<00:00, 21.71s/batch, val_loss=2.97]


Saved new best model(s) - Loss: 2.9741

Epoch 2 Summary:
Train - Loss: 2.8118, Acc: 15.22%, Precision: 0.2269, Recall: 0.1522, F1: 0.1347
Valid - Loss: 2.9741, Acc: 0.00%, Precision: 0.0000, Recall: 0.0000, F1: 0.0000

Epoch 3/20


...

Saved new best model(s) - Loss: 2.8216

Epoch 18 Summary:
Train - Loss: 1.5778, Acc: 39.13%, Precision: 0.3930, Recall: 0.3913, F1: 0.3201
Valid - Loss: 2.8216, Acc: 13.04%, Precision: 0.1304, Recall: 0.1304, F1: 0.1159

Epoch 19/20


Validation: 100%|██████████| 1/1 [00:19<00:00, 19.51s/batch, val_loss=2.78]


Saved new best model(s) - Loss: 2.7765

Epoch 19 Summary:
Train - Loss: 1.6467, Acc: 35.87%, Precision: 0.3447, Recall: 0.3587, F1: 0.3052
Valid - Loss: 2.7765, Acc: 13.04%, Precision: 0.1449, Recall: 0.1304, F1: 0.1275

Epoch 20/20


Validation: 100%|██████████| 1/1 [00:18<00:00, 18.73s/batch, val_loss=2.75]


Saved new best model(s) - Loss: 2.7452

Epoch 20 Summary:
Train - Loss: 1.6714, Acc: 39.13%, Precision: 0.3956, Recall: 0.3913, F1: 0.3368
Valid - Loss: 2.7452, Acc: 13.04%, Precision: 0.1304, Recall: 0.1304, F1: 0.1159

Training completed!
Results location: D:\yandex-ml-2025\results\TEST
Saving results...
saved metrics

Plotting metrics per class...
saved distribustion
